# WP1.1 - Three-phase power grids tests
This notebook demonstrates the operation of the multiconductor power flow with fully three-phase power grids.
The proof is based on the tests defined in the WP1.1 specification document.
All results have been compared against openDSS to confirm their correctness.

Here the needed libraries and functions used for the tests are imported.

In [1]:
import multiconductor as mc
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.12f' % x)
import warnings
import multiconductor.test.testing_toolbox
warnings.filterwarnings("ignore", category=FutureWarning)  
warnings.filterwarnings("ignore", category=DeprecationWarning) 

load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\zlib1.dll...
load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\abseil_dll.dll...
load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\utf8_validity.dll...
load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\re2.dll...
load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\libprotobuf.dll...
load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\highs.dll...
load C:\Users\pmohanty\miniconda\envs\pronto_environment\lib\site-packages\ortools\.libs\ortools.dll...


In [2]:
grids = multiconductor.test.testing_toolbox.load_all_test_grids("wp1_1_mc3ph")
l = []
for name, net in grids.items():
    mc.run_pf(net)
    c = multiconductor.test.testing_toolbox.compare_to_dss(net)
    l.append(c)
pd.DataFrame(l, index=grids.keys()).sort_index(key=lambda x: x.str.len())

,max_vm_pu_diff,max_va_degree_diff,max_i_from_ka_diff,max_i_to_ka_diff,max_p_from_mw_diff,max_p_to_mw_diff,max_q_from_mvar_diff,max_q_to_mvar_diff
two_bus_symmetric_load.pkl,0.000000000113,0.000000019367,0.000000000336,0.000000000336,0.000000000000,0.000000000000,0.000000000001,0.000000000000
four_bus_mesh_symmetric.pkl,0.000000000077,0.000000026887,0.000000000313,0.000000000313,0.000000000000,0.000000000000,0.000000000002,0.000000000002
two_bus_asymmetric_load.pkl,0.000000002974,0.000000019367,0.000000017212,0.000000017212,0.000000003320,0.000000003305,0.000000007292,0.000000007237
two_bus_symmetric_sgens.pkl,0.000000000166,0.000000030498,0.000000000607,0.000000000607,0.000000000001,0.000000000003,0.000000000004,0.000000000001
four_bus_mesh_asymmetric.pkl,0.000000000352,0.000000030832,0.000000004775,0.000000004775,0.000000000897,0.000000000886,0.000000000141,0.000000000145
two_bus_asymmetric_sgens.pkl,0.000000000178,0.000000031479,0.000000000932,0.000000000932,0.000000000075,0.000000000078,0.000000000008,0.000000000014
four_bus_radial_symmetric.pkl,0.000000000166,0.000000032886,0.000000000675,0.000000000675,0.000000000020,0.000000000020,0.000000000001,0.000000000004
five_bus_mesh_symmetric_Dy.pkl,0.000000001645,0.000000164095,0.000000000251,0.000000000251,0.000000000002,0.000000000003,0.000000000003,0.000000000002
four_bus_radial_asymmetric.pkl,0.000000089197,0.000000200847,0.000000267187,0.000000267190,0.000000050859,0.000000050299,0.000000133081,0.000000131991
five_bus_mesh_symmetric_Yy.pkl,0.000000001612,0.000000167336,0.000000000248,0.000000000248,0.000000000002,0.000000000003,0.000000000003,0.000000000002


## Tests on 2-bus grid
These tests serve to demonstrate the correct operation of different types of loads or generators, in both symmetric and asymmetric conditions and with the simplest possible grid configuration.

### 2-bus grid - symmetric loads

In [3]:
net = grids["two_bus_symmetric_load.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.1287881740429384e-10
Max overall va_degree difference: 1.9366762490301284e-08
Max overall i_from_ka difference: 3.3621026795938036e-10
Max overall i_to_ka difference: 3.362012820917748e-10
Max overall p_from_mw difference: 4.702193495687013e-13
Max overall p_to_mw difference: 2.402869569984034e-13
Max overall q_from_mvar difference: 1.3033645343552003e-12
Max overall q_to_mvar difference: 1.209293082338192e-13

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999998 0.999999999998   -0.000000000179    0.000000019188
      2     0.999999999998 0.999999999998 -120.000000000179 -119.999999987268
      3     0.999999999998 0.999999999998  119.999999999821  120.000000006276
1     0     0.000000000000 0.000000000000  108.970407808487   33.27304048

### 2-bus grid - asymmetric loads

In [4]:
net = grids["two_bus_asymmetric_load.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 2.9736607620733935e-09
Max overall va_degree difference: 1.93667626314508e-08
Max overall i_from_ka difference: 1.7212079442274764e-08
Max overall i_to_ka difference: 1.7212324510129662e-08
Max overall p_from_mw difference: 3.3196831984716457e-09
Max overall p_to_mw difference: 3.3053130094051753e-09
Max overall q_from_mvar difference: 7.292491866449369e-09
Max overall q_to_mvar difference: 7.2373746700521435e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999998 0.999999999998   -0.000000000179    0.000000019188
      2     0.999999999999 0.999999999999 -120.000000000373 -119.999999987462
      3     1.000000000002 1.000000000002  119.999999999704  120.000000006159
1     0     0.000000036316 0.000000036316 -142.575574970215 -142.5755940

### 2-bus grid - symmetric generators

In [5]:
net = grids["two_bus_symmetric_sgens.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6564971616617186e-10
Max overall va_degree difference: 3.049846974967352e-08
Max overall i_from_ka difference: 6.068594932129301e-10
Max overall i_to_ka difference: 6.068655439284143e-10
Max overall p_from_mw difference: 1.0904818714685405e-12
Max overall p_to_mw difference: 3.434842688054829e-12
Max overall q_from_mvar difference: 3.876377517586516e-12
Max overall q_to_mvar difference: 8.351973626585796e-13

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000002 1.000000000002    0.000000000366    0.000000019732
      2     1.000000000002 1.000000000002 -119.999999999634 -119.999999986723
      3     1.000000000002 1.000000000002  120.000000000366  120.000000006821
1     0     0.000000000000 0.000000000000  -66.801409486352 -133.6622385001

### 2-bus grid - asymmetric generators

In [6]:
net = grids["two_bus_asymmetric_sgens.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.7812684660611922e-10
Max overall va_degree difference: 3.1478784190408504e-08
Max overall i_from_ka difference: 9.317011745446635e-10
Max overall i_to_ka difference: 9.317044358247983e-10
Max overall p_from_mw difference: 7.528334552975302e-11
Max overall p_to_mw difference: 7.812135313645108e-11
Max overall q_from_mvar difference: 8.425582280480182e-12
Max overall q_to_mvar difference: 1.4071091514189504e-11

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000002 1.000000000002    0.000000000366    0.000000019732
      2     1.000000000001 1.000000000001 -119.999999999816 -119.999999986905
      3     0.999999999999 0.999999999999  120.000000000287  120.000000006743
1     0     0.000000006134 0.000000006134  160.353002384682  160.353002913

## Tests on 4-bus grid

The following tests are used to demonstrate the correct operation of various three-phase grids with different loads or generation types with a radial and meshed 4-bus grid configuration. 

### 4-bus grid - radial - symmetric loads and generators

In [7]:
net = grids["four_bus_radial_symmetric.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6612244913005725e-10
Max overall va_degree difference: 3.288643807275449e-08
Max overall i_from_ka difference: 6.75212816214632e-10
Max overall i_to_ka difference: 6.752149533939544e-10
Max overall p_from_mw difference: 2.0289009364173223e-11
Max overall p_to_mw difference: 2.0438141901547163e-11
Max overall q_from_mvar difference: 1.4771751530306965e-12
Max overall q_to_mvar difference: 4.276801135461028e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999998 0.999999999998    0.000000000008    0.000000019375
      2     0.999999999998 0.999999999998 -119.999999999992 -119.999999987081
      3     0.999999999998 0.999999999998  120.000000000008  120.000000006463
1     0     0.000000000000 0.000000000000   69.758547525707   33.489309967

### 4-bus grid - radial - asymmetric loads and generators

In [8]:
net = grids["four_bus_radial_asymmetric.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 8.919661431683323e-08
Max overall va_degree difference: 2.0084739560388698e-07
Max overall i_from_ka difference: 2.6718718215690274e-07
Max overall i_to_ka difference: 2.671902443185381e-07
Max overall p_from_mw difference: 5.0859476314474295e-08
Max overall p_to_mw difference: 5.029922679408716e-08
Max overall q_from_mvar difference: 1.3308062046368113e-07
Max overall q_to_mvar difference: 1.3199093491528952e-07

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999998 0.999999999998    0.000000000008    0.000000019375
      2     0.999999999998 0.999999999998 -120.000000000563 -119.999999987652
      3     1.000000000003 1.000000000003  119.999999999694  120.000000006149
1     0     0.000000036346 0.000000036346 -142.817544806718 -142.8177066

### 4-bus grid - meshed - symmetric loads and generators

In [9]:
net = grids["four_bus_mesh_symmetric.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 7.743938823523422e-11
Max overall va_degree difference: 2.6887409465548018e-08
Max overall i_from_ka difference: 3.1258792082544673e-10
Max overall i_to_ka difference: 3.12584146067163e-10
Max overall p_from_mw difference: 2.3326653109112527e-13
Max overall p_to_mw difference: 4.1096119562933353e-13
Max overall q_from_mvar difference: 2.3037448684815054e-12
Max overall q_to_mvar difference: 2.0002766804028127e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999998 0.999999999998    0.000000000009    0.000000019375
      2     0.999999999998 0.999999999998 -119.999999999991 -119.999999987080
      3     0.999999999998 0.999999999998  120.000000000009  120.000000006464
1     0     0.000000000000 0.000000000000   93.219494896853   33.4887715

### 4-bus grid - meshed - asymmetric loads and generators

In [10]:
net = grids["four_bus_mesh_asymmetric.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 3.5178637780575173e-10
Max overall va_degree difference: 3.083194144437584e-08
Max overall i_from_ka difference: 4.775151080105289e-09
Max overall i_to_ka difference: 4.775148304547727e-09
Max overall p_from_mw difference: 8.965240667901941e-10
Max overall p_to_mw difference: 8.864030794086819e-10
Max overall q_from_mvar difference: 1.406563841094277e-10
Max overall q_to_mvar difference: 1.4490564130553718e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999998 0.999999999998    0.000000000009    0.000000019376
      2     0.999999999998 0.999999999998 -120.000000000563 -119.999999987651
      3     1.000000000003 1.000000000003  119.999999999695  120.000000006151
1     0     0.000000036344 0.000000036344 -142.817464833426 -142.8174645513

## Test on 5-bus grid

The following tests are used to demonstrate the correct operation of various three-phase grids, with different loads and generators, with a radial 5-bus grid configuration, which represents the 4-bus radial grid with a transformer. 

### 5-bus grid - radial - symmetric loads and generators - Dyn transformer

In [11]:
net = grids["five_bus_radial_symmetric_Dy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.7278840580559063e-09
Max overall va_degree difference: 1.6863743468320536e-07
Max overall i_from_ka difference: 4.573450784750577e-10
Max overall i_to_ka difference: 4.57348450777495e-10
Max overall p_from_mw difference: 4.011435281170428e-12
Max overall p_to_mw difference: 3.5325371100514147e-12
Max overall q_from_mvar difference: 6.547021551197435e-12
Max overall q_to_mvar difference: 5.5845738732192285e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003233    0.000000021077
      2     0.999999999453 0.999999999726 -119.999999996767 -119.999999985378
      3     0.999999999453 0.999999999726  120.000000003233  120.000000008166
1     0     0.000000000000 0.000000000000  171.924001297670  163.344377708

### 5-bus grid - radial - asymmetric loads and generators - Dyn transformer

In [12]:
net = grids["five_bus_radial_symmetric_Dy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.7278840580559063e-09
Max overall va_degree difference: 1.6863743468320536e-07
Max overall i_from_ka difference: 4.573450784750577e-10
Max overall i_to_ka difference: 4.57348450777495e-10
Max overall p_from_mw difference: 4.011435281170428e-12
Max overall p_to_mw difference: 3.5325371100514147e-12
Max overall q_from_mvar difference: 6.547021551197435e-12
Max overall q_to_mvar difference: 5.5845738732192285e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003233    0.000000021077
      2     0.999999999453 0.999999999726 -119.999999996767 -119.999999985378
      3     0.999999999453 0.999999999726  120.000000003233  120.000000008166
1     0     0.000000000000 0.000000000000  171.924001297670  163.344377708

### 5-bus grid - radial - symmetric loads and generators - Yy transformer

In [13]:
net = grids["five_bus_radial_symmetric_Yy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6954035952920776e-09
Max overall va_degree difference: 1.7188400669443382e-07
Max overall i_from_ka difference: 4.5167808382373664e-10
Max overall i_to_ka difference: 4.5168160878183983e-10
Max overall p_from_mw difference: 4.001937670139455e-12
Max overall p_to_mw difference: 3.524890448969309e-12
Max overall q_from_mvar difference: 6.529000322157141e-12
Max overall q_to_mvar difference: 5.5694784992666585e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000 -179.997144972359    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003233    0.000000021077
      2     0.999999999453 0.999999999726 -119.999999996767 -119.999999985378
      3     0.999999999453 0.999999999726  120.000000003233  120.000000008166
1     0     0.000000000000 0.000000000000  -43.220827896619   54.9094256

### 5-bus grid - radial - asymmetric loads and generators - Yy transformer

In [14]:
net = grids["five_bus_radial_asymmetric_Yy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.926176107502897e-09
Max overall va_degree difference: 1.8661280593024188e-07
Max overall i_from_ka difference: 3.6415166576597713e-09
Max overall i_to_ka difference: 3.6415205295625697e-09
Max overall p_from_mw difference: 7.185595105041465e-10
Max overall p_to_mw difference: 7.259309403595537e-10
Max overall q_from_mvar difference: 1.5591787409785507e-10
Max overall q_to_mvar difference: 1.70475704733275e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000064 0.000000000000 -149.958757260724    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003234    0.000000021077
      2     0.999999999452 0.999999999726 -119.999999997142 -119.999999985754
      3     0.999999999454 0.999999999727  120.000000003038  120.000000007970
1     0     0.000000040017 0.000000040017   30.040549764547   30.040550055

### 5-bus grid - meshed - symmetric loads and generators - Dy transformer

In [15]:
net = grids["five_bus_mesh_symmetric_Dy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6449097639537058e-09
Max overall va_degree difference: 1.640952760340042e-07
Max overall i_from_ka difference: 2.5134896941558793e-10
Max overall i_to_ka difference: 2.513578165053154e-10
Max overall p_from_mw difference: 2.206675814298009e-12
Max overall p_to_mw difference: 2.722192263271417e-12
Max overall q_from_mvar difference: 2.7134019044226104e-12
Max overall q_to_mvar difference: 1.6959426213641188e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999453 0.999999999727    0.000000003234    0.000000021078
      2     0.999999999453 0.999999999727 -119.999999996766 -119.999999985378
      3     0.999999999453 0.999999999727  120.000000003234  120.000000008167
1     0     0.000000000000 0.000000000000   46.701400395999  159.66987818

### 5-bus grid - meshed - asymmetric loads and generators - Dy transformer

In [16]:
net = grids["five_bus_mesh_asymmetric_Dy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6849808215368967e-09
Max overall va_degree difference: 1.629397630154017e-07
Max overall i_from_ka difference: 5.759487609058311e-10
Max overall i_to_ka difference: 5.759515919745439e-10
Max overall p_from_mw difference: 8.034089712949388e-11
Max overall p_to_mw difference: 8.118584624017267e-11
Max overall q_from_mvar difference: 2.719346839721859e-11
Max overall q_to_mvar difference: 2.8890563986277984e-11

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999452 0.999999999726    0.000000003120    0.000000020964
      2     0.999999999455 0.999999999728 -119.999999996965 -119.999999985576
      3     0.999999999452 0.999999999726  120.000000002976  120.000000007909
1     0     0.000000040020 0.000000040020   60.032497333846   60.0324977584

### 5-bus grid - meshed - symmetric loads and generators - Yy transformer

In [17]:
net = grids["five_bus_mesh_symmetric_Yy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6123760104846951e-09
Max overall va_degree difference: 1.6733622748565935e-07
Max overall i_from_ka difference: 2.484150801729257e-10
Max overall i_to_ka difference: 2.48419680659584e-10
Max overall p_from_mw difference: 2.2055239579099606e-12
Max overall p_to_mw difference: 2.7216388864825802e-12
Max overall q_from_mvar difference: 2.705163052113893e-12
Max overall q_to_mvar difference: 1.6898177600038408e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000 -179.997144972359    0.000000000000
      1     0.999999999453 0.999999999727    0.000000003234    0.000000021078
      2     0.999999999453 0.999999999727 -119.999999996766 -119.999999985377
      3     0.999999999453 0.999999999727  120.000000003234  120.000000008167
1     0     0.000000000000 0.000000000000  -18.776873116202   53.41183788

### 5-bus grid - meshed - asymmetric loads and generators - Yy transformer

In [18]:
net = grids["five_bus_mesh_asymmetric_Yy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.5972820843757063e-09
Max overall va_degree difference: 1.6438757105552426e-07
Max overall i_from_ka difference: 5.797056515377541e-10
Max overall i_to_ka difference: 5.797035421140073e-10
Max overall p_from_mw difference: 8.033346730884627e-11
Max overall p_to_mw difference: 8.118002971235772e-11
Max overall q_from_mvar difference: 2.71885448183129e-11
Max overall q_to_mvar difference: 2.889235118513911e-11

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000064 0.000000000000 -149.966809691757    0.000000000000
      1     0.999999999453 0.999999999727    0.000000003234    0.000000021078
      2     0.999999999452 0.999999999726 -119.999999997142 -119.999999985753
      3     0.999999999454 0.999999999728  120.000000003039  120.000000007972
1     0     0.000000040020 0.000000040020   30.032497333371   30.03249776069

## 20-bus grid 
The following tests are used to demonstrate the correct operation of three-phase grids in multiconductor with a radial 20-bus grid.

### 20-bus grid - symmetric loads and generators - Dy transformer

In [19]:
net = grids["twenty_bus_radial_symmetric_Dy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6818648695959837e-09
Max overall va_degree difference: 1.630632908700136e-07
Max overall i_from_ka difference: 4.812405890453064e-10
Max overall i_to_ka difference: 4.812453352487367e-10
Max overall p_from_mw difference: 1.2407158633820359e-12
Max overall p_to_mw difference: 3.3503373686460947e-12
Max overall q_from_mvar difference: 3.444389738704867e-12
Max overall q_to_mvar difference: 2.7334228492906053e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003234    0.000000021077
      2     0.999999999453 0.999999999726 -119.999999996766 -119.999999985378
      3     0.999999999453 0.999999999726  120.000000003234  120.000000008166
1     0     0.000000000000 0.000000000000 -121.910874615314 -178.22067346

### 20-bus grid - asymmetric loads and generators - Dy transformer

In [20]:
net = grids["twenty_bus_radial_asymmetric_Dy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6945105318910691e-09
Max overall va_degree difference: 1.6870001218194375e-07
Max overall i_from_ka difference: 7.986282885008578e-10
Max overall i_to_ka difference: 7.986338812493443e-10
Max overall p_from_mw difference: 7.568275520175582e-11
Max overall p_to_mw difference: 7.828920497998659e-11
Max overall q_from_mvar difference: 1.0339756828514624e-11
Max overall q_to_mvar difference: 1.4112629467821769e-11

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999452 0.999999999726    0.000000003119    0.000000020963
      2     0.999999999454 0.999999999728 -119.999999996965 -119.999999985577
      3     0.999999999452 0.999999999726  120.000000002975  120.000000007907
1     0     0.000000039989 0.000000039989   60.027046495293   60.02704687

### 20-bus grid - symmetric loads and generators - Yy transformer

In [21]:
net = grids["twenty_bus_radial_symmetric_Yy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6494137167200051e-09
Max overall va_degree difference: 1.6630317317689602e-07
Max overall i_from_ka difference: 4.755385529797707e-10
Max overall i_to_ka difference: 4.755427718272642e-10
Max overall p_from_mw difference: 1.2378917335631456e-12
Max overall p_to_mw difference: 3.3499834850569954e-12
Max overall q_from_mvar difference: 3.434078542363661e-12
Max overall q_to_mvar difference: 2.6711468709327034e-12

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000   98.007583884934    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003234    0.000000021077
      2     0.999999999453 0.999999999726 -119.999999996766 -119.999999985378
      3     0.999999999453 0.999999999726  120.000000003234  120.000000008166
1     0     0.000000000000 0.000000000000  -88.220825725868   86.4234982

### 20-bus grid - asymmetric loads and generators - Yy transformer

In [22]:
net = grids["twenty_bus_radial_asymmetric_Yy.pkl"]
mc.run_pf(net)
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.6763672672226448e-09
Max overall va_degree difference: 1.6852897033459158e-07
Max overall i_from_ka difference: 8.063632539467847e-10
Max overall i_to_ka difference: 8.063649331591094e-10
Max overall p_from_mw difference: 7.567047682899286e-11
Max overall p_to_mw difference: 7.828836884327117e-11
Max overall q_from_mvar difference: 1.034707129005108e-11
Max overall q_to_mvar difference: 1.4113031923668196e-11

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000064 0.000000000000 -149.972260539470    0.000000000000
      1     0.999999999453 0.999999999726    0.000000003234    0.000000021078
      2     0.999999999452 0.999999999726 -119.999999997142 -119.999999985754
      3     0.999999999454 0.999999999727  120.000000003038  120.000000007970
1     0     0.000000039989 0.000000039989   30.027046489298   30.027046878